In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('data/TransactionReportWithWeather.csv')

# Event Name: Clean
# Event ID: Clean
# Event Timestamp: Clean (Converted to DateTime)
df['Event Timestamp'] = pd.to_datetime(df['Event Timestamp'], utc=True).dt.tz_convert('America/New_York')
# Venue Name: Clean
# Transaction Timestamp: Clean (Converted to DateTime)
df['Transaction Timestamp'] = pd.to_datetime(df['Transaction Timestamp'], utc=True).dt.tz_convert('America/New_York')
# External Ref: Dropped (All values were nan)
df.drop(columns = ['External Ref'], inplace = True)
# Price Scale: Clean
# Section: Clean
# Row: Clean
# Seats: Clean (All seats in transaction are listed, but not worth fixing for analysis)
# Primary Seat Ids: Clean (All seats in transaction are listed, but not worth fixing for analysis)
# Average Unit Price: Clean (Removed $ and converted to float)
df['Average Unit Price'] = df['Average Unit Price'].str.replace('$', '').astype(float)
# Sales Total: Clean (Removed $ and converted to float)
df['Sales Total'] = df['Sales Total'].str.replace('$', '').astype(float)
# Tickets Sold: Clean
# Market Place: Clean
# Transaction Type: Clean (But has both purchase and return transactions)
# Seat Created At: Dropped (Not useful for analysis)
# Seat Modified At: Dropped (Not useful for analysis)
df.drop(columns = ['Seat Created At', 'Seat Modified At'], inplace = True)
# Highest Inventory Price: Clean (Removed $ and converted to float)
df['Highest Inventory Price'] = df['Highest Inventory Price'].str.replace('$', '').astype(float)
# Lowest Inventory Price: Clean (Removed $ and converted to float)
df['Lowest Inventory Price'] = df['Lowest Inventory Price'].str.replace('$', '').astype(float)
# Initial Inventory Price: Clean (Removed $ and converted to float)
df['Initial Inventory Price'] = df['Initial Inventory Price'].str.replace('$', '').astype(float)
# Event Score: Clean
# Customer First Name, Last Name, Email, Phone: All dropped (All nan)
df.drop(columns = ['Customer First Name', 'Customer Last Name', 'Customer Email', 'Customer Phone'], inplace = True)
# weather_code: Dropped (Added weather_condition, which is more useful)
# temperature_c: Dropped (Added temperature_f, which is more useful)
df.drop(columns = ['weather_code', 'temperature_c'], inplace = True)
# weather_condition: Clean
# temperature_f: Clean
# precipitation: Clean
# humidity: Clean
# weather_category: Clean


In [3]:
# ── Remove purchase/return pairs ──────────────────────────────────────────────

purchases = df[df['Transaction Type'].str.lower() == 'purchase'].copy()
returns   = df[df['Transaction Type'].str.lower() == 'return'].copy()

match_cols = ['Event ID', 'Primary Seat IDs']

# Sort both by transaction time so ordering is chronological
purchases = purchases.sort_values('Transaction Timestamp').copy()
returns   = returns.sort_values('Transaction Timestamp').copy()

# Count how many returns exist for each (Event ID, Seat) key
return_counts = returns.groupby(match_cols).size().rename('return_count')

# Join return counts onto purchases
purchases = purchases.join(return_counts, on=match_cols)
purchases['return_count'] = purchases['return_count'].fillna(0).astype(int)

# cumcount is now assigned in chronological order — earliest purchases
# are marked for removal first (they are the ones that were returned)
purchases['_cumcount'] = purchases.groupby(match_cols).cumcount()
purchases['_drop'] = purchases['_cumcount'] < purchases['return_count']

df_clean = purchases[~purchases['_drop']].drop(
    columns=['return_count', '_cumcount', '_drop']
).reset_index(drop=True)

print(f"Original rows      : {len(df)}")
print(f"Purchases          : {len(purchases)}")
print(f"Returns            : {len(returns)}")
print(f"Matched pairs      : {purchases['_drop'].sum()}")
print(f"Unmatched purchases: {len(df_clean)}")

Original rows      : 18410
Purchases          : 18066
Returns            : 344
Matched pairs      : 311
Unmatched purchases: 17755


In [4]:
# Calculate time difference between purchase and event
df_clean['Days Before Event'] = (df_clean['Event Timestamp'] - df_clean['Transaction Timestamp']).dt.days

In [5]:
negative_one = df_clean[df_clean['Days Before Event'] == -1].copy()

negative_one['Hours After Start'] = (
    negative_one['Transaction Timestamp'] - negative_one['Event Timestamp']
).dt.total_seconds() / 3600

within_3hrs = negative_one[negative_one['Hours After Start'] <= 3]

print(f"Total -1 transactions      : {len(negative_one)}")
print(f"Within 3 hours of start    : {len(within_3hrs)}")
print(f"Percentage                 : {len(within_3hrs) / len(negative_one) * 100:.1f}%")

Total -1 transactions      : 1125
Within 3 hours of start    : 1122
Percentage                 : 99.7%


In [6]:
df_clean = df_clean[df_clean['Days Before Event'] != -26] # Weird outlier that is likely a data error, so we will remove it
df_clean['Days Before Event'] = df_clean['Days Before Event'].replace(-1, 0) # Game-day purchases should include post-start purchases, so we will replace -1 with 0

In [7]:
# Add start time
df_clean['Start Time'] = df_clean['Event Timestamp'].dt.strftime('%-I:%M %p')

In [8]:
df_clean.to_csv('./data/FinalTransactionReport.csv', index = False)